# 02 · XGBoost con selección de columnas y tratamiento de placeholders

Cambios respecto del baseline (`01_baseline_logistica`):

1. **Modelo:** XGBoost con categóricas nativas (sin one-hot, sin imputar, sin escalar).
2. **Columnas eliminadas (61):** auto (2), datos de contacto (6), características del edificio (47) y consultas al Credit Bureau (6). El detalle está en `src/features.py`.
3. **Placeholders:** `DAYS_EMPLOYED = 365243` y `ORGANIZATION_TYPE = 'XNA'` marcan a las mismas 55.374 filas (jubilados, más 22 desempleados). Se reemplazan por:
   * `FLAG_JUBILADO = 1`
   * `DAYS_EMPLOYED = NaN` (el valor falso de 1000 años desaparece)
   * `ORGANIZATION_TYPE = 'Jubilado'`

Para respaldar la eliminación de columnas con datos, se compara el mismo XGBoost **con y sin** las columnas eliminadas, en los mismos folds.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import ID_COL, TARGET_COL
from src.cv import run_cv
from src.data import attach_folds, load_application
from src.features_app import DROP_COLS, preparar
from src.models import build_xgboost
from src.submission import make_submission

## 1. Datos y preprocesamiento

In [2]:
train = load_application("train")
test = load_application("test")
folds = attach_folds(train)
y = train[TARGET_COL]

X, X_test = preparar(train, test)
print(f"Columnas eliminadas: {len(DROP_COLS)} | features finales: {X.shape[1]}")
print(f"Jubilados marcados: {X['FLAG_JUBILADO'].sum():,} "
      f"(default {y[X['FLAG_JUBILADO'] == 1].mean():.2%} vs {y[X['FLAG_JUBILADO'] == 0].mean():.2%} del resto)")

Columnas eliminadas: 61 | features finales: 60
Jubilados marcados: 55,374 (default 5.40% vs 8.66% del resto)


## 2. XGBoost con la selección de columnas

In [3]:
res = run_cv(build_xgboost, X, y, folds, X_test=X_test)

Fold 0: AUC = 0.7550


Fold 1: AUC = 0.7640


Fold 2: AUC = 0.7563


Fold 3: AUC = 0.7634


Fold 4: AUC = 0.7546


AUC OOF total: 0.7586 | media por fold: 0.7587 ± 0.0042


## 3. Comparación: mismo modelo sin eliminar columnas

Mismo preprocesamiento de placeholders y mismos hiperparámetros; lo único que cambia es que no se elimina ninguna columna.

In [4]:
X_full, _ = preparar(train, test, drop_cols=())
res_full = run_cv(build_xgboost, X_full, y, folds, verbose=False)

comparacion = pd.DataFrame({
    "features": [X_full.shape[1], X.shape[1]],
    "AUC OOF": [res_full["oof_auc"], res["oof_auc"]],
    "AUC por fold (media)": [pd.Series(res_full["fold_aucs"]).mean(), pd.Series(res["fold_aucs"]).mean()],
}, index=["Todas las columnas", "Con selección"]).round(4)
comparacion

,features,AUC OOF,AUC por fold (media)
Todas las columnas,121,0.7605,0.7605
Con selección,60,0.7586,0.7587


## 4. Importancia de variables (gain), modelo entrenado sin el fold 0

Primera aproximación a la interpretabilidad; más adelante conviene sumar permutación / SHAP.

In [5]:
modelo = build_xgboost().fit(X[folds != 0], y[folds != 0])
imp = pd.Series(modelo.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
(imp / imp.sum()).head(20).round(3).to_frame("gain relativo")

,gain relativo
EXT_SOURCE_2,0.105
EXT_SOURCE_3,0.094
NAME_EDUCATION_TYPE,0.051
FLAG_DOCUMENT_3,0.043
EXT_SOURCE_1,0.038
CODE_GENDER,0.034
DAYS_EMPLOYED,0.029
REGION_RATING_CLIENT,0.028
NAME_CONTRACT_TYPE,0.027
REGION_RATING_CLIENT_W_CITY,0.026


## 5. Submission

In [6]:
path = make_submission(test[ID_COL], res["test_pred"], name="xgb_seleccion", cv_auc=res["oof_auc"])

Submission guardada en /home/claude/tp1-homecredit/submissions/20260921_2306_xgb_seleccion_cv0.7586.csv (48744 filas)


## Notas

* Los hiperparámetros son un punto de partida (600 árboles, learning rate 0,03, profundidad 5). Falta la optimización de hiperparámetros sobre los folds.
* La cantidad de árboles se eligió mirando el fold 0; en la búsqueda de hiperparámetros hay que elegirla sin mirar validación (por ejemplo, con early stopping dentro del train de cada fold).